In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

In [2]:
class PrunableLinear(nn.Module):

    def __init__(self, in_features: int, out_features: int):
        super().__init__()
        self.in_features  = in_features
        self.out_features = out_features

        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        self.bias   = nn.Parameter(torch.zeros(out_features))
        nn.init.kaiming_uniform_(self.weight, a=np.sqrt(5))

        self.gate_scores = nn.Parameter(torch.zeros(out_features, in_features))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        gates         = torch.sigmoid(self.gate_scores)   # ∈ (0, 1)
        pruned_weights = self.weight * gates               # element-wise product
        return F.linear(x, pruned_weights, self.bias)

    def get_gates(self) -> torch.Tensor:
        """Return the current gate values (detached from graph)."""
        return torch.sigmoid(self.gate_scores).detach()

    def extra_repr(self) -> str:
        return f"in_features={self.in_features}, out_features={self.out_features}"

In [5]:
# Network definition
class SelfPruningNet(nn.Module):
  def __init__(self, hidden_sizes=(512, 256, 128)):
          super().__init__()
          layers = []
          in_dim = 3 * 32 * 32  # CIFAR-10 flattened

          for h in hidden_sizes:
              layers.append(PrunableLinear(in_dim, h))
              layers.append(nn.BatchNorm1d(h))
              layers.append(nn.ReLU())
              layers.append(nn.Dropout(0.3))
              in_dim = h

          layers.append(PrunableLinear(in_dim, 10))  # output layer
          self.net = nn.Sequential(*layers)

  def forward(self, x: torch.Tensor) -> torch.Tensor:
          x = x.view(x.size(0), -1)   # flatten
          return self.net(x)

  def prunable_layers(self):
          """Yield all PrunableLinear modules in the network."""
          for m in self.modules():
              if isinstance(m, PrunableLinear):
                  yield m

In [6]:
#Sparsity Loss
def sparsity_loss(model: SelfPruningNet) -> torch.Tensor:

    total = torch.tensor(0.0, requires_grad=True)
    for layer in model.prunable_layers():
        gates = torch.sigmoid(layer.gate_scores)
        total = total + gates.sum()
    return total

In [14]:
def get_loaders(batch_size: int = 128):
    transform_train = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomCrop(32, padding=4),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465),
                             (0.2023, 0.1994, 0.2010)),
    ])
    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465),
                             (0.2023, 0.1994, 0.2010)),
    ])
    train_ds = datasets.CIFAR10(root='./data', train=True,
                                download=True, transform=transform_train)
    test_ds  = datasets.CIFAR10(root='./data', train=False,
                                download=True, transform=transform_test)
    train_loader = DataLoader(train_ds, batch_size=batch_size,
                              shuffle=True,  num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=256,
                              shuffle=False, num_workers=2, pin_memory=True)
    return train_loader, test_loader

def train_one_epoch(model, loader, optimizer, device, lam: float):
    model.train()
    total_loss = 0.0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        clf_loss  = F.cross_entropy(logits, y)
        spar_loss = sparsity_loss(model)
        loss = clf_loss + lam * spar_loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    correct = total = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        preds = model(x).argmax(dim=1)
        correct += (preds == y).sum().item()
        total   += y.size(0)
    return 100.0 * correct / total


@torch.no_grad()
def compute_sparsity(model, threshold: float = 1e-2) -> float:
    """Fraction of gates whose value is below threshold (treated as pruned)."""
    all_gates = torch.cat([torch.sigmoid(l.gate_scores).flatten()
                           for l in model.prunable_layers()])
    pruned = (all_gates < threshold).sum().item()
    return 100.0 * pruned / all_gates.numel()


def run_experiment(lam: float, epochs: int, device, train_loader, test_loader):
    print(f"\n{'='*50}")
    print(f"  λ = {lam}  |  epochs = {epochs}")
    print(f"{'='*50}")

    model = SelfPruningNet(hidden_sizes=(512, 256, 128)).to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    for epoch in range(1, epochs + 1):
        loss = train_one_epoch(model, train_loader, optimizer, device, lam)
        scheduler.step()
        if epoch % 5 == 0 or epoch == epochs:
            acc      = evaluate(model, test_loader, device)
            sparsity = compute_sparsity(model)
            print(f"  Epoch {epoch:3d} | loss={loss:.4f} | "
                  f"acc={acc:.2f}% | sparsity={sparsity:.1f}%")

    final_acc      = evaluate(model, test_loader, device)
    final_sparsity = compute_sparsity(model)
    print(f"\n  ✓ Final → Accuracy: {final_acc:.2f}%  |  Sparsity: {final_sparsity:.1f}%")
    return model, final_acc, final_sparsity


def plot_gate_distribution(model, lam: float, save_path: str):
    """Plot histogram of all final gate values for a trained model."""
    all_gates = torch.cat([torch.sigmoid(l.gate_scores).flatten()
                           for l in model.prunable_layers()]).detach().cpu().numpy()

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.hist(all_gates, bins=100, color='steelblue', edgecolor='none', alpha=0.85)
    ax.set_xlabel('Gate Value', fontsize=12)
    ax.set_ylabel('Count', fontsize=12)
    ax.set_title(f'Gate Value Distribution  (λ = {lam})', fontsize=13)

    near_zero = (all_gates < 0.01).sum()
    ax.annotate(f'{near_zero:,} gates\nnear zero',
                xy=(0.01, ax.get_ylim()[1] * 0.8),
                fontsize=10, color='crimson')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f"  Gate distribution plot saved → {save_path}")

In [9]:
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    EPOCHS     = 30
    BATCH_SIZE = 128
    LAMBDAS    = [1e-5, 1e-4, 1e-3]   # low → medium → high sparsity pressure

    train_loader, test_loader = get_loaders(BATCH_SIZE)

    results = {}
    best_model = None
    best_lam   = LAMBDAS[1]   # track "medium" lambda as the best model for plotting

    for lam in LAMBDAS:
        model, acc, sparsity = run_experiment(
            lam, EPOCHS, device, train_loader, test_loader
        )
        results[lam] = {"accuracy": acc, "sparsity": sparsity}
        if lam == best_lam:
            best_model = model

Device: cuda


100%|██████████| 170M/170M [00:03<00:00, 43.4MB/s]



  λ = 1e-05  |  epochs = 30
  Epoch   5 | loss=10.1870 | acc=47.50% | sparsity=0.0%
  Epoch  10 | loss=10.1155 | acc=50.56% | sparsity=0.0%
  Epoch  15 | loss=10.0660 | acc=52.59% | sparsity=0.0%
  Epoch  20 | loss=10.0083 | acc=54.19% | sparsity=0.0%
  Epoch  25 | loss=9.9663 | acc=55.53% | sparsity=0.0%
  Epoch  30 | loss=9.9481 | acc=56.10% | sparsity=0.0%

  ✓ Final → Accuracy: 56.10%  |  Sparsity: 0.0%

  λ = 0.0001  |  epochs = 30
  Epoch   5 | loss=77.8779 | acc=48.40% | sparsity=0.0%
  Epoch  10 | loss=77.7977 | acc=50.05% | sparsity=0.0%
  Epoch  15 | loss=77.7414 | acc=53.20% | sparsity=0.0%
  Epoch  20 | loss=77.6858 | acc=55.02% | sparsity=0.0%
  Epoch  25 | loss=77.6451 | acc=56.14% | sparsity=0.0%
  Epoch  30 | loss=77.6252 | acc=56.63% | sparsity=0.0%

  ✓ Final → Accuracy: 56.63%  |  Sparsity: 0.0%

  λ = 0.001  |  epochs = 30
  Epoch   5 | loss=389.1536 | acc=48.13% | sparsity=0.0%
  Epoch  10 | loss=320.5093 | acc=50.77% | sparsity=0.0%
  Epoch  15 | loss=319.7593 | 

In [10]:
print("\n\n" + "="*55)
print(f"{'Lambda':<12} {'Test Accuracy':>15} {'Sparsity (%)':>14}")
print("-"*55)
for lam, r in results.items():
    print(f"{lam:<12} {r['accuracy']:>13.2f}%  {r['sparsity']:>12.1f}%")
print("="*55)



Lambda         Test Accuracy   Sparsity (%)
-------------------------------------------------------
1e-05                56.10%           0.0%
0.0001               56.63%           0.0%
0.001                56.26%           0.0%
